In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CARREGAR DATASET_FEATURES_V3
# ============================================================
df = pd.read_parquet('../data/gold/dataset_features_v3.parquet')

print(f"dataset_features_v3 carregado: {df.shape}")
print(f"Período: {df['data'].min().date()} → {df['data'].max().date()}")
print(f"Features: {df.shape[1]}")
print(f"\nNovas features nowcasting:")
novas = ['fator_nowcasting', 'casos_nowcast', 'municipio_id', 
         'casos_por_100k', 'casos_nowcast_por_100k']
for f in novas:
    print(f" {f}: {df[f].describe().to_dict()['mean']:.4f} (média)")

# Verificar nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
print(f"\nColunas com nulos: {len(nulos)}")
if len(nulos) > 0:
    print(nulos)

dataset_features_v3 carregado: (2242, 60)
Período: 2018-02-12 → 2024-12-28
Features: 60

Novas features nowcasting:
 fator_nowcasting: 1.6974 (média)
 casos_nowcast: 139.6538 (média)
 municipio_id: 5103403.0000 (média)
 casos_por_100k: 10.5059 (média)
 casos_nowcast_por_100k: 22.0803 (média)

Colunas com nulos: 2
radiacao_lag_28d    28
radiacao_mm_14d     13
dtype: int64


In [2]:
# ============================================================
# 2. PREPARAR FEATURES E TARGET
# ============================================================

# Remover nulos (mesma decisão defensável do v2)
df_clean = df.dropna().copy()
print(f"Após remover nulos: {df_clean.shape} (removidos {len(df)-len(df_clean)} registros)")

# Separar features e target
# Remover colunas não-features
drop_cols = ['data', 'casos', 'casos_nowcast', 'municipio_id']

X = df_clean.drop(columns=drop_cols)
y = df_clean['casos']

print(f"\nFeatures para treino: {X.shape[1]}")
print(f"Target: casos (min={y.min()}, max={y.max()}, média={y.mean():.1f})")

# Garantir ordem temporal
df_clean = df_clean.sort_values('data').reset_index(drop=True)
X = df_clean.drop(columns=drop_cols)
y = df_clean['casos']

# Comparar com v2 — quantas features a mais?
print(f"\nv2: 55 features → v3: {X.shape[1]} features (+{X.shape[1]-51} novas efetivas)")
print(f"\nFeatures v3:")
print(X.columns.tolist())

Após remover nulos: (2214, 60) (removidos 28 registros)

Features para treino: 56
Target: casos (min=1, max=440, média=66.8)

v2: 55 features → v3: 56 features (+5 novas efetivas)

Features v3:
['precipitacao_total', 'temp_media', 'temp_max', 'temp_min', 'umidade_media', 'umidade_max', 'umidade_min', 'ano', 'mes', 'semana_ano', 'dia_ano', 'trimestre', 'mes_seno', 'mes_cosseno', 'semana_seno', 'semana_cosseno', 'casos_lag_7d', 'casos_lag_14d', 'casos_lag_21d', 'casos_lag_28d', 'casos_mm_7d', 'casos_mm_14d', 'casos_mm_28d', 'precip_lag_28d', 'umidade_lag_28d', 'temp_lag_28d', 'precip_lag_35d', 'umidade_lag_35d', 'temp_lag_35d', 'precip_lag_42d', 'umidade_lag_42d', 'temp_lag_42d', 'precip_mm_7d', 'umidade_mm_7d', 'precip_mm_14d', 'umidade_mm_14d', 'precip_mm_28d', 'umidade_mm_28d', 'amplitude_termica', 'precip_acum_7d', 'precip_acum_14d', 'precip_acum_28d', 'dias_sem_chuva', 'ndvi', 'ndwi', 'anos_desde_pico', 'ciclo_epidemico', 'casos_acum_ano', 'radiacao_mj', 'radiacao_lag_28d', 'radiaca

In [3]:
# ============================================================
# 3. RETREINO ROLLING WINDOW LIGHTGBM V3
# ============================================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

tscv = TimeSeriesSplit(n_splits=5)

params_lgbm = {
    'objective': 'regression',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

resultados = []
modelos_fold = []

print("Treinando LightGBM v3 com TimeSeriesSplit...")
print("="*55)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**params_lgbm)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    y_pred = model.predict(X_val)
    y_pred = np.maximum(y_pred, 0)
    
    mae  = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2   = r2_score(y_val, y_pred)
    
    resultados.append({'fold': fold, 'mae': mae, 'rmse': rmse, 'r2': r2,
                       'n_train': len(X_train), 'n_val': len(X_val)})
    modelos_fold.append(model)
    print(f"Fold {fold}: MAE={mae:.1f} | RMSE={rmse:.1f} | R²={r2:.3f} | train={len(X_train)} val={len(X_val)}")

res_df = pd.DataFrame(resultados)
res_sem_fold1 = res_df[res_df['fold'] > 1]

print("="*55)
print(f"\nResultado v3 (Folds 2-5):")
print(f"  MAE:  {res_sem_fold1['mae'].mean():.1f} ± {res_sem_fold1['mae'].std():.1f}")
print(f"  RMSE: {res_sem_fold1['rmse'].mean():.1f} ± {res_sem_fold1['rmse'].std():.1f}")
print(f"  R²:   {res_sem_fold1['r2'].mean():.3f} ± {res_sem_fold1['r2'].std():.3f}")

print(f"\nComparação v2 vs v3:")
print(f"  v2 — MAE: 17.4 | RMSE: 27.8 | R²: 0.830")
print(f"  v3 — MAE: {res_sem_fold1['mae'].mean():.1f} | RMSE: {res_sem_fold1['rmse'].mean():.1f} | R²: {res_sem_fold1['r2'].mean():.3f}")

Treinando LightGBM v3 com TimeSeriesSplit...
Fold 1: MAE=48.7 | RMSE=89.8 | R²=0.009 | train=369 val=369
Fold 2: MAE=1.3 | RMSE=2.1 | R²=0.997 | train=738 val=369
Fold 3: MAE=1.8 | RMSE=3.2 | R²=0.998 | train=1107 val=369
Fold 4: MAE=1.5 | RMSE=3.0 | R²=0.998 | train=1476 val=369
Fold 5: MAE=5.5 | RMSE=16.8 | R²=0.977 | train=1845 val=369

Resultado v3 (Folds 2-5):
  MAE:  2.5 ± 2.0
  RMSE: 6.3 ± 7.0
  R²:   0.992 ± 0.010

Comparação v2 vs v3:
  v2 — MAE: 17.4 | RMSE: 27.8 | R²: 0.830
  v3 — MAE: 2.5 | RMSE: 6.3 | R²: 0.992


In [4]:
# ============================================================
# 4. CORRIGIR DATA LEAKAGE — remover features derivadas do target
# ============================================================

# Features que causam leakage — derivadas diretamente de 'casos'
leakage_cols = ['casos_por_100k', 'casos_nowcast_por_100k', 'fator_nowcasting']

# fator_nowcasting também é suspeito — é uma função da semana do ano
# que o modelo pode usar para inferir o volume de casos

X_clean = X.drop(columns=leakage_cols)

print(f"Features antes: {X.shape[1]}")
print(f"Features após remover leakage: {X_clean.shape[1]}")
print(f"Removidas: {leakage_cols}")

# Retreinar sem leakage
resultados_v3b = []

print("\nRetreinando sem data leakage...")
print("="*55)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_clean), 1):
    X_train, X_val = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**params_lgbm)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    y_pred = model.predict(X_val)
    y_pred = np.maximum(y_pred, 0)
    
    mae  = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2   = r2_score(y_val, y_pred)
    
    resultados_v3b.append({'fold': fold, 'mae': mae, 'rmse': rmse, 'r2': r2})
    print(f"Fold {fold}: MAE={mae:.1f} | RMSE={rmse:.1f} | R²={r2:.3f}")

res_v3b = pd.DataFrame(resultados_v3b)
res_v3b_sem1 = res_v3b[res_v3b['fold'] > 1]

print("="*55)
print(f"\nv3b — sem leakage (Folds 2-5):")
print(f"  MAE:  {res_v3b_sem1['mae'].mean():.1f} ± {res_v3b_sem1['mae'].std():.1f}")
print(f"  RMSE: {res_v3b_sem1['rmse'].mean():.1f} ± {res_v3b_sem1['rmse'].std():.1f}")
print(f"  R²:   {res_v3b_sem1['r2'].mean():.3f} ± {res_v3b_sem1['r2'].std():.3f}")

print(f"\nComparação final:")
print(f"  v2  — MAE: 17.4 | RMSE: 27.8 | R²: 0.830")
print(f"  v3b — MAE: {res_v3b_sem1['mae'].mean():.1f} | RMSE: {res_v3b_sem1['rmse'].mean():.1f} | R²: {res_v3b_sem1['r2'].mean():.3f}")

Features antes: 56
Features após remover leakage: 53
Removidas: ['casos_por_100k', 'casos_nowcast_por_100k', 'fator_nowcasting']

Retreinando sem data leakage...
Fold 1: MAE=55.9 | RMSE=93.5 | R²=-0.075
Fold 2: MAE=11.3 | RMSE=17.8 | R²=0.762
Fold 3: MAE=18.7 | RMSE=29.1 | R²=0.799
Fold 4: MAE=15.8 | RMSE=26.2 | R²=0.855
Fold 5: MAE=24.5 | RMSE=40.8 | R²=0.863

v3b — sem leakage (Folds 2-5):
  MAE:  17.6 ± 5.5
  RMSE: 28.5 ± 9.5
  R²:   0.820 ± 0.048

Comparação final:
  v2  — MAE: 17.4 | RMSE: 27.8 | R²: 0.830
  v3b — MAE: 17.6 | RMSE: 28.5 | R²: 0.820


In [5]:
# ============================================================
# 5. OTIMIZAÇÃO OPTUNA — v3b
# ============================================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Usar Fold 4 como validação para otimização (mais representativo)
fold_idx = list(tscv.split(X_clean))[3]  # Fold 4
X_tr, X_vl = X_clean.iloc[fold_idx[0]], X_clean.iloc[fold_idx[1]]
y_tr, y_vl = y.iloc[fold_idx[0]], y.iloc[fold_idx[1]]

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    return mean_absolute_error(y_vl, np.maximum(model.predict(X_vl), 0))

print("Otimizando hiperparâmetros (50 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nMelhor MAE no Fold 4: {study.best_value:.2f}")
print(f"Melhores parâmetros: {study.best_params}")

Otimizando hiperparâmetros (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Melhor MAE no Fold 4: 15.49
Melhores parâmetros: {'learning_rate': 0.04993437617040339, 'num_leaves': 72, 'min_child_samples': 23, 'subsample': 0.8758480010430004, 'colsample_bytree': 0.6407712390490528, 'reg_alpha': 0.4442512608406364, 'reg_lambda': 0.7694877363002522}


In [6]:
# ============================================================
# 6. RETREINO FINAL COM MELHORES PARAMS + MAPE
# ============================================================

def mean_absolute_percentage_error(y_true, y_pred):
    """MAPE robusto — ignora zeros no denominador"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true > 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Parâmetros otimizados pelo Optuna
best_params = {
    'objective': 'regression',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'random_state': 42,
    **study.best_params
}

resultados_final = []
modelos_final = []

print("Retreino final — LightGBM v3 otimizado (MAE + RMSE + MAPE + R²)")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_clean), 1):
    X_train, X_val = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**best_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    y_pred = np.maximum(model.predict(X_val), 0)
    
    mae  = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2   = r2_score(y_val, y_pred)
    mape = mean_absolute_percentage_error(y_val, y_pred)
    
    resultados_final.append({
        'fold': fold, 'mae': mae, 'rmse': rmse, 
        'r2': r2, 'mape': mape
    })
    modelos_final.append(model)
    print(f"Fold {fold}: MAE={mae:.1f} | RMSE={rmse:.1f} | R²={r2:.3f} | MAPE={mape:.1f}%")

res_final = pd.DataFrame(resultados_final)
res_f25 = res_final[res_final['fold'] > 1]

print("="*60)
print(f"\nLightGBM v3 otimizado — Folds 2-5:")
print(f"  MAE:  {res_f25['mae'].mean():.1f} ± {res_f25['mae'].std():.1f} casos/dia")
print(f"  RMSE: {res_f25['rmse'].mean():.1f} ± {res_f25['rmse'].std():.1f} casos/dia")
print(f"  MAPE: {res_f25['mape'].mean():.1f} ± {res_f25['mape'].std():.1f}%")
print(f"  R²:   {res_f25['r2'].mean():.3f} ± {res_f25['r2'].std():.3f}")

print(f"\nComparação v2 vs v3 otimizado:")
print(f"  v2  — MAE: 17.4 | RMSE: 27.8 | R²: 0.830 | MAPE: N/A")
print(f"  v3  — MAE: {res_f25['mae'].mean():.1f} | RMSE: {res_f25['rmse'].mean():.1f} | R²: {res_f25['r2'].mean():.3f} | MAPE: {res_f25['mape'].mean():.1f}%")

Retreino final — LightGBM v3 otimizado (MAE + RMSE + MAPE + R²)
Fold 1: MAE=56.0 | RMSE=93.6 | R²=-0.079 | MAPE=54.6%
Fold 2: MAE=11.1 | RMSE=16.9 | R²=0.785 | MAPE=51.2%
Fold 3: MAE=19.1 | RMSE=28.3 | R²=0.810 | MAPE=67.3%
Fold 4: MAE=15.5 | RMSE=26.1 | R²=0.856 | MAPE=34.3%
Fold 5: MAE=24.2 | RMSE=40.2 | R²=0.867 | MAPE=39.6%

LightGBM v3 otimizado — Folds 2-5:
  MAE:  17.5 ± 5.6 casos/dia
  RMSE: 27.9 ± 9.6 casos/dia
  MAPE: 48.1 ± 14.6%
  R²:   0.829 ± 0.039

Comparação v2 vs v3 otimizado:
  v2  — MAE: 17.4 | RMSE: 27.8 | R²: 0.830 | MAPE: N/A
  v3  — MAE: 17.5 | RMSE: 27.9 | R²: 0.829 | MAPE: 48.1%


In [7]:
# ============================================================
# 7. DIAGNÓSTICO DO MAPE ALTO
# ============================================================

# Pegar predições do Fold 4 (mais representativo)
fold_idx = list(tscv.split(X_clean))[3]
X_tr, X_vl = X_clean.iloc[fold_idx[0]], X_clean.iloc[fold_idx[1]]
y_tr, y_vl = y.iloc[fold_idx[0]], y.iloc[fold_idx[1]]

model_f4 = modelos_final[3]
y_pred_f4 = np.maximum(model_f4.predict(X_vl), 0)

df_diag = pd.DataFrame({
    'real': y_vl.values,
    'pred': y_pred_f4,
    'erro_abs': np.abs(y_vl.values - y_pred_f4),
    'erro_pct': np.abs(y_vl.values - y_pred_f4) / np.where(y_vl.values > 0, y_vl.values, 1) * 100
})

print("Distribuição dos casos reais no Fold 4:")
print(f"  Mín: {y_vl.min()} | Mediana: {y_vl.median():.0f} | Máx: {y_vl.max()}")
print(f"  Casos <= 10: {(y_vl <= 10).sum()} registros")
print(f"  Casos <= 20: {(y_vl <= 20).sum()} registros")
print(f"  Casos > 100: {(y_vl > 100).sum()} registros")

print(f"\nMAPE por faixa de casos reais:")
for faixa, mask in [
    ('1-10 casos',   (y_vl >= 1)  & (y_vl <= 10)),
    ('11-50 casos',  (y_vl >= 11) & (y_vl <= 50)),
    ('51-150 casos', (y_vl >= 51) & (y_vl <= 150)),
    ('>150 casos',   (y_vl > 150))
]:
    if mask.sum() > 0:
        mape_faixa = df_diag[mask.values]['erro_pct'].mean()
        print(f"  {faixa}: MAPE={mape_faixa:.1f}% (n={mask.sum()})")

print(f"\nConclusão: MAPE alto é causado por:")
baixos = (y_vl <= 10).sum()
print(f"  {baixos} dias com ≤ 10 casos — pequeno denominador infla o percentual")
print(f"  Isso é esperado e documentado na literatura para séries com zeros/baixos valores")

Distribuição dos casos reais no Fold 4:
  Mín: 2 | Mediana: 42 | Máx: 287
  Casos <= 10: 31 registros
  Casos <= 20: 69 registros
  Casos > 100: 100 registros

MAPE por faixa de casos reais:
  1-10 casos: MAPE=137.8% (n=31)
  11-50 casos: MAPE=28.5% (n=177)
  51-150 casos: MAPE=23.4% (n=97)
  >150 casos: MAPE=16.4% (n=64)

Conclusão: MAPE alto é causado por:
  31 dias com ≤ 10 casos — pequeno denominador infla o percentual
  Isso é esperado e documentado na literatura para séries com zeros/baixos valores


In [8]:
# ============================================================
# 8. sMAPE + TABELA FINAL DE MÉTRICAS PARA O ARTIGO
# ============================================================

def smape(y_true, y_pred):
    """sMAPE — simétrico, robusto a valores baixos"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denominador = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominador > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominador[mask]) * 100

# Recalcular todas as métricas com sMAPE
resultados_smape = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_clean), 1):
    X_train, X_val = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    y_pred = np.maximum(modelos_final[fold-1].predict(X_val), 0)
    
    resultados_smape.append({
        'fold': fold,
        'mae':   mean_absolute_error(y_val, y_pred),
        'rmse':  np.sqrt(mean_squared_error(y_val, y_pred)),
        'r2':    r2_score(y_val, y_pred),
        'mape':  mean_absolute_percentage_error(y_val, y_pred),
        'smape': smape(y_val, y_pred)
    })

res_sm = pd.DataFrame(resultados_smape)
res_sm25 = res_sm[res_sm['fold'] > 1]

print("="*60)
print("TABELA FINAL DE MÉTRICAS — LightGBM v3 (Folds 2-5)")
print("Para uso no artigo/resumo expandido")
print("="*60)
print(f"\n{'Métrica':<10} {'Média':>10} {'±DP':>10} {'Interpretação'}")
print(f"{'-'*55}")
print(f"{'MAE':<10} {res_sm25['mae'].mean():>10.1f} {res_sm25['mae'].std():>9.1f}  casos/dia")
print(f"{'RMSE':<10} {res_sm25['rmse'].mean():>10.1f} {res_sm25['rmse'].std():>9.1f}  casos/dia")
print(f"{'R²':<10} {res_sm25['r2'].mean():>10.3f} {res_sm25['r2'].std():>9.3f}  [0-1]")
print(f"{'MAPE':<10} {res_sm25['mape'].mean():>10.1f}% {res_sm25['mape'].std():>8.1f}% distorcido por valores baixos")
print(f"{'sMAPE':<10} {res_sm25['smape'].mean():>10.1f}% {res_sm25['smape'].std():>8.1f}% métrica recomendada")

print(f"\nComparação completa v2 vs v3:")
print(f"{'Modelo':<20} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'sMAPE':>8}")
print(f"{'-'*55}")
print(f"{'LightGBM v2':<20} {'17.4':>8} {'27.8':>8} {'0.830':>8} {'N/A':>8}")
print(f"{'LightGBM v3':<20} {res_sm25['mae'].mean():>8.1f} {res_sm25['rmse'].mean():>8.1f} {res_sm25['r2'].mean():>8.3f} {res_sm25['smape'].mean():>7.1f}%")

TABELA FINAL DE MÉTRICAS — LightGBM v3 (Folds 2-5)
Para uso no artigo/resumo expandido

Métrica         Média        ±DP Interpretação
-------------------------------------------------------
MAE              17.5       5.6  casos/dia
RMSE             27.9       9.6  casos/dia
R²              0.829     0.039  [0-1]
MAPE             48.1%     14.6% distorcido por valores baixos
sMAPE            32.4%      5.5% métrica recomendada

Comparação completa v2 vs v3:
Modelo                    MAE     RMSE       R²    sMAPE
-------------------------------------------------------
LightGBM v2              17.4     27.8    0.830      N/A
LightGBM v3              17.5     27.9    0.829    32.4%


In [9]:
# ============================================================
# 9. SALVAR MODELO + MÉTRICAS
# ============================================================
from pathlib import Path
import json

Path('../models').mkdir(exist_ok=True)

# Salvar modelo do Fold 5 (maior conjunto de treino)
modelo_prod = modelos_final[4]
joblib.dump(modelo_prod, '../models/lgbm_v3_producao.pkl')

# Salvar métricas completas
res_sm.to_csv('../reports/metricas_lgbm_v3.csv', index=False)

# Salvar resumo para o artigo
resumo = {
    'modelo': 'LightGBM v3 otimizado (Optuna)',
    'dataset': 'dataset_features_v3',
    'n_features': int(X_clean.shape[1]),
    'n_registros': int(len(df_clean)),
    'periodo': '2018-02-12 a 2024-12-28',
    'validacao': 'TimeSeriesSplit 5 folds (Folds 2-5)',
    'metricas_folds_2_5': {
        'MAE_media': round(res_sm25['mae'].mean(), 1),
        'MAE_dp': round(res_sm25['mae'].std(), 1),
        'RMSE_media': round(res_sm25['rmse'].mean(), 1),
        'RMSE_dp': round(res_sm25['rmse'].std(), 1),
        'R2_media': round(res_sm25['r2'].mean(), 3),
        'R2_dp': round(res_sm25['r2'].std(), 3),
        'sMAPE_media': round(res_sm25['smape'].mean(), 1),
        'sMAPE_dp': round(res_sm25['smape'].std(), 1),
    },
    'best_params': study.best_params
}

with open('../reports/resumo_metricas_v3.json', 'w', encoding='utf-8') as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

print("Modelo salvo: models/lgbm_v3_producao.pkl")
print("Métricas salvas: reports/metricas_lgbm_v3.csv")
print("Resumo JSON salvo: reports/resumo_metricas_v3.json")
print(f"\nModelo de produção: LightGBM v3")
print(f"   MAE:   {res_sm25['mae'].mean():.1f} ± {res_sm25['mae'].std():.1f} casos/dia")
print(f"   RMSE:  {res_sm25['rmse'].mean():.1f} ± {res_sm25['rmse'].std():.1f} casos/dia")
print(f"   R²:    {res_sm25['r2'].mean():.3f} ± {res_sm25['r2'].std():.3f}")
print(f"   sMAPE: {res_sm25['smape'].mean():.1f} ± {res_sm25['smape'].std():.1f}%")

Modelo salvo: models/lgbm_v3_producao.pkl
Métricas salvas: reports/metricas_lgbm_v3.csv
Resumo JSON salvo: reports/resumo_metricas_v3.json

Modelo de produção: LightGBM v3
   MAE:   17.5 ± 5.6 casos/dia
   RMSE:  27.9 ± 9.6 casos/dia
   R²:    0.829 ± 0.039
   sMAPE: 32.4 ± 5.5%
